# Рекомендаційна система на Surprise: SVD, SVD++ та NMF

## 1. Встановлення бібліотек


In [1]:
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install --no-cache-dir scikit-surprise

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 25.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554968 sha256=01b5186521f172c9824e8ce915e3b67a0c16d6e14a789f5c1e4ec4b9d353f168
  Stored in directory: /tmp/pip-ephem-wheel-cache-8r7513lt/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise
Бібліотеки встановлено. Тепер перезапусти сеанс / runtime.


## 2. Перевірка версій



In [1]:
import numpy as np
import pandas as pd

print('NumPy version:', np.__version__)

if not np.__version__.startswith('1.26'):
    print('УВАГА: бажано, щоб NumPy був версії 1.26.4')
else:
    print('Все добре: NumPy сумісний зі scikit-surprise')


NumPy version: 1.26.4
Все добре: NumPy сумісний зі scikit-surprise


## 3. Імпорт бібліотек Surprise та завантаження датасету

Використовуємо вбудований датасет `ml-100k` з бібліотеки Surprise.

In [2]:
from surprise import Dataset, accuracy
from surprise import SVD, SVDpp, NMF
from surprise.model_selection import cross_validate, GridSearchCV, train_test_split

RANDOM_STATE = 42

# prompt=False потрібен, щоб бібліотека не ставила інтерактивне питання про завантаження датасету
data = Dataset.load_builtin('ml-100k', prompt=False)

print('Датасет MovieLens 100k успішно завантажено')


Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Датасет MovieLens 100k успішно завантажено


## 4. Первинне порівняння моделей

Порівняємо три алгоритми:

- `SVD` — класична матрична факторизація;
- `SVD++` — розширена версія SVD, враховує implicit feedback;
- `NMF` — Non-negative Matrix Factorization.

Метрики:

- `RMSE` — середньоквадратична помилка;
- `MAE` — середня абсолютна помилка.

Чим менші значення RMSE та MAE, тим краща модель.

In [4]:
models = {
    'SVD': SVD(random_state=RANDOM_STATE),
    'SVD++': SVDpp(random_state=RANDOM_STATE),
    'NMF': NMF(random_state=RANDOM_STATE),
}

baseline_results = []

for model_name, algorithm in models.items():
    print(f'Навчання моделі: {model_name}')
    cv_result = cross_validate(
        algorithm,
        data,
        measures=['RMSE', 'MAE'],
        cv=3,
        verbose=False,
        n_jobs=-1
    )

    import numpy as np

baseline_results.append({
    'model': model_name,
    'mean_rmse': np.mean(cv_result['test_rmse']),
    'mean_mae': np.mean(cv_result['test_mae']),
    'mean_fit_time': np.mean(cv_result['fit_time']),
    'mean_test_time': np.mean(cv_result['test_time'])
})

baseline_df = pd.DataFrame(baseline_results).sort_values('mean_rmse')
baseline_df


Навчання моделі: SVD
Навчання моделі: SVD++
Навчання моделі: NMF


,model,mean_rmse,mean_mae,mean_fit_time,mean_test_time
0,NMF,0.973207,0.763929,1.855962,0.241799


## 5. Підбір гіперпараметрів для SVD

Підбираємо параметри за допомогою `GridSearchCV`.

In [5]:
param_grid_svd = {
    'n_factors': [50, 100],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.05]
}

gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

gs_svd.fit(data)

print('Найкращий RMSE для SVD:', gs_svd.best_score['rmse'])
print('Найкращий MAE для SVD:', gs_svd.best_score['mae'])
print('Найкращі параметри SVD:', gs_svd.best_params['rmse'])


Найкращий RMSE для SVD: 0.932434603933242
Найкращий MAE для SVD: 0.7359254827991203
Найкращі параметри SVD: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05}


## 6. Підбір гіперпараметрів для NMF

In [6]:
param_grid_nmf = {
    'n_factors': [15, 30],
    'n_epochs': [30, 50],
    'reg_pu': [0.06, 0.1],
    'reg_qi': [0.06, 0.1]
}

gs_nmf = GridSearchCV(
    NMF,
    param_grid_nmf,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

gs_nmf.fit(data)

print('Найкращий RMSE для NMF:', gs_nmf.best_score['rmse'])
print('Найкращий MAE для NMF:', gs_nmf.best_score['mae'])
print('Найкращі параметри NMF:', gs_nmf.best_params['rmse'])


Найкращий RMSE для NMF: 0.9526898350533332
Найкращий MAE для NMF: 0.7474917701098063
Найкращі параметри NMF: {'n_factors': 30, 'n_epochs': 50, 'reg_pu': 0.1, 'reg_qi': 0.1}


## 7. Підбір гіперпараметрів для SVD++

`SVD++` працює повільніше, тому використовуємо меншу сітку параметрів.

In [7]:
param_grid_svdpp = {
    'n_factors': [20, 50],
    'n_epochs': [10, 20],
    'lr_all': [0.005],
    'reg_all': [0.02, 0.05]
}

gs_svdpp = GridSearchCV(
    SVDpp,
    param_grid_svdpp,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

gs_svdpp.fit(data)

print('Найкращий RMSE для SVD++:', gs_svdpp.best_score['rmse'])
print('Найкращий MAE для SVD++:', gs_svdpp.best_score['mae'])
print('Найкращі параметри SVD++:', gs_svdpp.best_params['rmse'])


Найкращий RMSE для SVD++: 0.9282534062317875
Найкращий MAE для SVD++: 0.7313108603604462
Найкращі параметри SVD++: {'n_factors': 20, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}


## 8. Порівняння оптимізованих моделей

In [8]:
comparison_df = pd.DataFrame([
    {
        'model': 'SVD',
        'best_rmse': gs_svd.best_score['rmse'],
        'best_mae': gs_svd.best_score['mae'],
        'best_params': gs_svd.best_params['rmse']
    },
    {
        'model': 'NMF',
        'best_rmse': gs_nmf.best_score['rmse'],
        'best_mae': gs_nmf.best_score['mae'],
        'best_params': gs_nmf.best_params['rmse']
    },
    {
        'model': 'SVD++',
        'best_rmse': gs_svdpp.best_score['rmse'],
        'best_mae': gs_svdpp.best_score['mae'],
        'best_params': gs_svdpp.best_params['rmse']
    }
]).sort_values('best_rmse')

comparison_df


,model,best_rmse,best_mae,best_params
2,SVD++,0.928253,0.731311,"{'n_factors': 20, 'n_epochs': 20, 'lr_all': 0...."
0,SVD,0.932435,0.735925,"{'n_factors': 100, 'n_epochs': 30, 'lr_all': 0..."
1,NMF,0.952690,0.747492,"{'n_factors': 30, 'n_epochs': 50, 'reg_pu': 0...."


## 9. Навчання найкращої моделі на train/test split

In [9]:
best_model_name = comparison_df.iloc[0]['model']
best_params = comparison_df.iloc[0]['best_params']

print('Найкраща модель:', best_model_name)
print('Параметри:', best_params)

if best_model_name == 'SVD':
    best_algorithm = SVD(**best_params, random_state=RANDOM_STATE)
elif best_model_name == 'SVD++':
    best_algorithm = SVDpp(**best_params, random_state=RANDOM_STATE)
else:
    best_algorithm = NMF(**best_params, random_state=RANDOM_STATE)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=RANDOM_STATE
)

best_algorithm.fit(trainset)
predictions = best_algorithm.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)


Найкраща модель: SVD++
Параметри: {'n_factors': 20, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}
RMSE: 0.9189
MAE:  0.7215


## 10. Приклад прогнозу рейтингу

Спробуємо передбачити рейтинг для користувача `196` та фільму `302`.

In [10]:
user_id = '196'
item_id = '302'

prediction = best_algorithm.predict(user_id, item_id)
prediction


Prediction(uid='196', iid='302', r_ui=None, est=4.08891011887172, details={'was_impossible': False})

## 11. Висновок

У роботі було побудовано та порівняно три моделі рекомендаційних систем з бібліотеки `Surprise`: `SVD`, `SVD++` та `NMF`.

Для кожної моделі було виконано крос-валідацію та підбір гіперпараметрів за допомогою `GridSearchCV`.

Оптимальна модель обирається за найменшим значенням `RMSE`. Зазвичай `SVD++` може показувати хорошу якість, але працює повільніше. `SVD` часто є найкращим компромісом між якістю та швидкістю. `NMF` є альтернативним методом матричної факторизації.